# Benchmark histórico de movimiento, alineado por vencimiento

## Objetivo

Construir un único benchmark histórico contemporáneo para cada par ya seleccionado. No se calcula qué predictor gana: esta notebook no crea diferencias, correlaciones, errores, rankings ni gráficos de desempeño contra `terminal_move_rel`.

## Regla fijada antes de ejecutar

Para un par en t con strike K y vencimiento T, `H` es la diferencia entre los índices del calendario de sesiones de t y T. `k = K / SPY_close_t`. Para cada final histórico `u ≤ t`, la ventana usa `abs(SPY_close_u / SPY_close_(u-H) - k)`. Se toman exactamente las últimas 252 ventanas completas.

Solo T aporta el horizonte de calendario; los cierres usados para el promedio terminan como máximo en t. Una ventana que contiene un split se detecta de forma conservadora; el benchmark de esa fila queda faltante hasta definir un tratamiento, sin incorporar silenciosamente el retorno afectado.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root / 'scripts'))
from build_historical_move_baseline import build_historical_move_baseline
print('Lógica reutilizable: scripts/build_historical_move_baseline.py')

Lógica reutilizable: scripts/build_historical_move_baseline.py


In [2]:
result = build_historical_move_baseline(write_csv=True)
aligned = result['aligned'].copy()
for column in ['observation_date', 'expiration', 'historical_first_window_start', 'historical_first_window_end',
               'historical_last_window_start', 'historical_last_window_end']:
    aligned[column] = pd.to_datetime(aligned[column])
print('Entrada:', result['input_path'])
print('Cierres sin ajustar:', result['price_path'])
print('CSV escrito:', result['output_path'])
display(pd.DataFrame([result['audit']]))

Entrada: C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\opciones_notas\straddle_ATM_SPY_volatilidad_futura\data\derived\spy_atm30_straddle_terminal_move_rel_2020-01-02_2026-08-31.csv
Cierres sin ajustar: C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\HMM_regime_options\data\derived\spy_daily_adjusted_2020_2026.csv
CSV escrito: C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\opciones_notas\straddle_ATM_SPY_volatilidad_futura\data\derived\spy_atm30_straddle_with_historical_move_baseline_2020-01-02_2026-08-31.csv


,rows_total,pairs_available,h_resolved_from_session_calendar,baseline_with_252_windows,fewer_than_252_windows,expiration_not_in_session_calendar,split_affected_rows,all_completed_windows_through_t,price_history_start,price_history_end
0,1674,1672,1652,1382,270,20,0,True,2020-01-02,2026-08-31


## Cobertura y controles de no anticipación

Las filas sin straddle y las censuradas se preservan. La disponibilidad del resultado terminal no decide si el benchmark se construye: lo determina la existencia de H en el calendario y de 252 ventanas que terminan en t o antes.

In [3]:
status_counts = (aligned['historical_baseline_absence_reason']
    .fillna('baseline_available')
    .value_counts()
    .rename_axis('baseline_status')
    .reset_index(name='rows'))
display(status_counts)

coverage_by_year = (aligned.assign(year=aligned['observation_date'].dt.year)
    .groupby('year', as_index=False)
    .agg(rows=('series_row_index', 'size'), pairs_available=('pair_available', 'sum'),
         h_resolved=('h_sessions', lambda values: values.notna().sum()),
         baseline_available=('historical_move_baseline_t', lambda values: values.notna().sum()),
         split_affected=('historical_split_affected_window_count', lambda values: (values > 0).sum())))
display(coverage_by_year)

completed = aligned['historical_move_baseline_t'].notna()
assert aligned.loc[completed, 'historical_window_count'].eq(252).all()
assert aligned.loc[completed, 'historical_dates_through_t_only'].all()
assert aligned.loc[completed, 'historical_last_window_end'].le(aligned.loc[completed, 'observation_date']).all()
print('Comprobación: todo cierre usado por un benchmark disponible tiene fecha ≤ t.')

,baseline_status,rows
0,baseline_available,1382
1,fewer_than_252_complete_historical_windows,270
2,expiration_not_in_available_session_calendar,20
3,no_straddle_available,2


,year,rows,pairs_available,h_resolved,baseline_available,split_affected
0,2020,253,252,252,0,0
1,2021,252,251,251,233,0
2,2022,251,251,251,251,0
3,2023,250,250,250,250,0
4,2024,252,252,252,252,0
5,2025,250,250,250,250,0
6,2026,166,166,146,146,0


Comprobación: todo cierre usado por un benchmark disponible tiene fecha ≤ t.


## Dos ejemplos de índices históricos

Se muestran el primer y último benchmark disponible. Los índices son cero-basados dentro del calendario local de sesiones; cada fila enseña H, k, las fechas de inicio/fin de la primera y última ventana de las 252 usadas, y verifica que la última termina en t o antes.

In [4]:
available = aligned.loc[aligned['historical_move_baseline_t'].notna()].copy()
examples = available.iloc[[0, -1]]
example_columns = [
    'series_row_index', 'observation_session_index', 'expiration_session_index',
    'observation_date', 'expiration', 'h_sessions', 'strike', 'spy_close_t',
    'k_strike_over_spot_t', 'historical_window_count',
    'historical_first_window_start', 'historical_first_window_end',
    'historical_last_window_start', 'historical_last_window_end',
    'historical_split_affected_window_count', 'historical_dates_through_t_only',
    'historical_move_baseline_t'
]
display(examples[example_columns])

,series_row_index,observation_session_index,expiration_session_index,observation_date,expiration,h_sessions,strike,spy_close_t,k_strike_over_spot_t,historical_window_count,historical_first_window_start,historical_first_window_end,historical_last_window_start,historical_last_window_end,historical_split_affected_window_count,historical_dates_through_t_only,historical_move_baseline_t
271,271,271.0,291.0,2021-01-29,2021-03-01,20.0,370.0,370.07,0.999811,252,2020-01-02,2020-01-31,2020-12-30,2021-01-29,0,True,0.061222
1653,1653,1653.0,1673.0,2026-08-03,2026-08-31,20.0,758.0,757.67,1.000436,252,2025-07-03,2025-08-01,2026-07-06,2026-08-03,0,True,0.025167


## Límites

Este benchmark es un promedio histórico descriptivo con horizonte H; no determina qué predictor es mejor. No se calculan diferencias, correlaciones, errores, rankings ni desempeño contra el movimiento terminal. La hora EOD de las cadenas sigue sin verificarse. El ejercicio americano y la falta de evidencia de ejecución simultánea siguen impidiendo interpretar prima menos valor intrínseco como P&L real.

## Mis notas

- 
- 
- 